# CNN Model for Image Classification

This notebook implements a CNN model using EfficientNetV2M for multi-class image classification.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet import preprocess_input as resnet_preprocess
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


## Setup and Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Data Preprocessing

Reorganize dataset from multiple folders into unified class structure.

In [ ]:
import os
import shutil
from pathlib import Path

source_dataset_dir = '/content/drive/MyDrive/data/new_data_14Oct'
unified_dataset_dir = '/content/drive/MyDrive/data/unified_dataset'

os.makedirs(unified_dataset_dir, exist_ok=True)

print("Scanning source directory structure...")
print(f"Source: {source_dataset_dir}\n")

main_folders = [f for f in os.listdir(source_dataset_dir)
                if os.path.isdir(os.path.join(source_dataset_dir, f))]

print(f"Found {len(main_folders)} main folders: {main_folders}\n")

class_counts = {}
total_images = 0

for main_folder in main_folders:
    main_folder_path = os.path.join(source_dataset_dir, main_folder)
    print(f"Processing folder: {main_folder}")

    class_folders = [f for f in os.listdir(main_folder_path)
                     if os.path.isdir(os.path.join(main_folder_path, f))]

    print(f"   Classes found: {class_folders}")

    for class_name in class_folders:
        class_source_path = os.path.join(main_folder_path, class_name)
        class_dest_path = os.path.join(unified_dataset_dir, class_name)

        os.makedirs(class_dest_path, exist_ok=True)

        image_files = [f for f in os.listdir(class_source_path)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif'))]

        copied_count = 0
        for img_file in image_files:
            source_img = os.path.join(class_source_path, img_file)
            dest_img_name = f"{main_folder}_{img_file}"
            dest_img = os.path.join(class_dest_path, dest_img_name)

            if not os.path.exists(dest_img):
                shutil.copy2(source_img, dest_img)
                copied_count += 1

            if class_name not in class_counts:
                class_counts[class_name] = 0
            class_counts[class_name] += 1
            total_images += 1

        print(f"   {class_name}: {copied_count} images copied ({len(image_files)} total)")

    print()

print("=" * 60)
print("DATASET REORGANIZATION COMPLETE")
print("=" * 60)
print(f"\nSummary:")
print(f"Total images processed: {total_images}")
print(f"Total classes: {len(class_counts)}")
print(f"\nImages per class:")
for class_name, count in sorted(class_counts.items()):
    print(f"   {class_name}: {count} images")

print(f"\nUnified dataset location: {unified_dataset_dir}")

In [ ]:
!ls /content/drive/MyDrive/data/unified_dataset

dataset_dir = '/content/drive/MyDrive/data/unified_dataset'
print(f"\nUsing dataset: {dataset_dir}")

## Data Generators

Create training and validation data generators with augmentation.

In [ ]:
dataset_dir = '/content/drive/MyDrive/data/unified_dataset'

img_size = (224, 224)
batch_size = 64

train_datagen = ImageDataGenerator(
    validation_split=0.2,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

validation_generator = val_datagen.flow_from_directory(
    dataset_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

num_classes = len(train_generator.class_indices)
print("Number of classes:", num_classes)
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")

## Dataset Exploration

Visualize class distribution and sample images.

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

print("Class indices (label → class name):")
for class_name, index in train_generator.class_indices.items():
    print(f"{index}: {class_name}")

print("\nSamples per class in training set:")
for cls in train_generator.class_indices.keys():
    cls_dir = os.path.join(dataset_dir, cls)
    count = len(os.listdir(cls_dir))
    print(f"{cls}: {count}")

print("\nShowing one sample per class:")
plt.figure(figsize=(12, 6))
for i, class_name in enumerate(train_generator.class_indices.keys()):
    class_dir = os.path.join(dataset_dir, class_name)
    sample_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not sample_files:
        print(f"No images found in {class_name}")
        continue
    sample_img_path = os.path.join(class_dir, sample_files[0])
    img = plt.imread(sample_img_path)
    plt.subplot(1, len(train_generator.class_indices), i + 1)
    plt.imshow(img)
    plt.title(class_name)
    plt.axis("off")

plt.tight_layout()
plt.show()

## Model Architecture

Build the model using EfficientNetV2M as the base with custom classification head.

In [ ]:
from tensorflow.keras.applications import EfficientNetV2M

base_model = EfficientNetV2M(
    weights='imagenet',
    include_top=False,
    input_shape=(img_size[0], img_size[1], 3),
    pooling='avg'
)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = layers.Dense(512, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = keras.Model(inputs=base_model.input, outputs=outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Training Callbacks

Configure callbacks for model checkpointing, early stopping, and learning rate reduction.

In [ ]:
checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

earlystop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    verbose=1
)

callbacks = [checkpoint, earlystop, reduce_lr]

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_labels = train_generator.classes
unique_classes = np.unique(class_labels)

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=class_labels
)

class_weights = dict(enumerate(class_weights_array))

print("Class distribution in training set:")
for class_name, class_idx in train_generator.class_indices.items():
    count = np.sum(class_labels == class_idx)
    weight = class_weights[class_idx]
    print(f"{class_name:12} | Samples: {count:4d} | Weight: {weight:.3f}")

## Initial Training

Train the model with frozen base layers.

In [ ]:
epochs = 50

history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator,
    callbacks=callbacks,
    class_weight=class_weights
)

## Fine-Tuning

Unfreeze the last layers of the base model for fine-tuning.

In [ ]:
for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

fine_tune_epochs = 0

history_fine = model.fit(
    train_generator,
    epochs=fine_tune_epochs,
    initial_epoch=history.epoch[-1],
    validation_data=validation_generator,
    callbacks=callbacks,
    class_weight=class_weights
)

## Training Visualization

Plot training and validation accuracy and loss curves.

In [ ]:
print("Keys in history.history:", history.history.keys())
print("Keys in history_fine.history:", history_fine.history.keys())

acc = history.history.get('accuracy', []) + history_fine.history.get('accuracy', [])
val_acc = history.history.get('val_accuracy', []) + history_fine.history.get('val_accuracy', [])
loss = history.history.get('loss', []) + history_fine.history.get('loss', [])
val_loss = history.history.get('val_loss', []) + history_fine.history.get('val_loss', [])

epochs_range = range(len(acc))

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training & Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training & Validation Loss')
plt.show()

In [ ]:
final_val_acc = val_acc[-1]
final_val_loss = val_loss[-1]
print(f"Final validation accuracy: {final_val_acc:.4f}")
print(f"Final validation loss: {final_val_loss:.4f}")

## Model Predictions

Display sample predictions from the validation set.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

class_indices = {v: k for k, v in train_generator.class_indices.items()}

x_batch, y_true = next(validation_generator)

num_samples = min(5, len(x_batch))

preds = model.predict(x_batch[:num_samples])
pred_labels = np.argmax(preds, axis=1)
true_labels = np.argmax(y_true[:num_samples], axis=1)

plt.figure(figsize=(15, 6))
for i in range(num_samples):
    plt.subplot(1, num_samples, i + 1)
    plt.imshow((x_batch[i] * 255).astype("uint8"))

    predicted_class = class_indices[pred_labels[i]]
    true_class = class_indices[true_labels[i]]
    confidence = np.max(preds[i]) * 100

    plt.title(
        f"P: {predicted_class}\nT: {true_class}\n({confidence:.1f}%)",
        fontsize=10,
        color=("green" if predicted_class == true_class else "red")
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

## Model Evaluation

Generate comprehensive evaluation metrics including classification report and confusion matrices.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

validation_generator.reset()
y_pred_probs = model.predict(validation_generator, steps=len(validation_generator), verbose=1)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true = validation_generator.classes

class_names = list(train_generator.class_indices.keys())
n_classes = len(class_names)

unique_labels = np.unique(y_true)
missing_labels = sorted(set(range(n_classes)) - set(unique_labels))
if missing_labels:
    print(f"Warning: these class indices are absent from validation set: {missing_labels}")

labels_for_report = list(range(n_classes))
report = classification_report(
    y_true,
    y_pred_classes,
    labels=labels_for_report,
    target_names=class_names,
    digits=4,
    zero_division=0
)
print(report)

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred_classes, labels=labels_for_report, zero_division=0
)
print("Per-class metrics:")
print(f"{'Class':<15} {'Prec':<7} {'Rec':<7} {'F1':<7} {'Support':<7}")
for i, name in enumerate(class_names):
    print(f"{name:<15} {precision[i]:<7.4f} {recall[i]:<7.4f} {f1[i]:<7.4f} {support[i]:<7}")

cm = confusion_matrix(y_true, y_pred_classes, labels=labels_for_report)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

row_sums = cm.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
cm_norm = cm.astype('float') / row_sums

plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Proportion'})
plt.title('Normalized Confusion Matrix (by true class)')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()